# Exercise 7.2 — Local vs Global Cost Functions in Deep Circuits

**Chapter 7: Quantum Machine Learning** &nbsp;|&nbsp; *Section 7.2: Trainability and Cost Function Design*

---

## Motivation

Not all cost functions are equally trainable.  A **global** cost (e.g., projecting onto the target state) and a **local** cost (e.g., measuring single-qubit observables) both suffer from barren plateaus in 2-design circuits — but the **rate** of exponential decay is dramatically different.  The local cost has gradient variance $\sim 1/(N \cdot 2^N)$, while the global cost decays as $\sim 1/2^{2N}$ — an exponential advantage for the local cost.  This exercise derives both scalings via the same Weingarten calculus and quantifies the advantage.

## Preliminaries / Toolbox

Before diving into the solution, recall the following concepts:

**1. Cost Functions in VQAs:** The objective is $C(\boldsymbol{\theta}) = \langle 0| U^\dagger(\boldsymbol{\theta}) O U(\boldsymbol{\theta}) |0\rangle$.

**2. Global vs. Local Observables:** A global cost relies on an observable that acts non-trivially on all qubits, e.g., $O_G = |0\rangle\langle 0|^{\otimes N}$. A local cost is a sum of local terms, e.g., $O_L = \frac{1}{N}\sum_j |0\rangle\langle 0|_j$.

**3. Barren Plateaus (2-design):** If $U(\boldsymbol{\theta})$ forms a 2-design, the variance of the cost function is $\mathrm{Var}[C] = \frac{\mathrm{Tr}(O^2)}{D(D+1)}$ where $D=2^N$.

**4. Trace properties:** $\mathrm{Tr}(A \otimes B) = \mathrm{Tr}(A)\mathrm{Tr}(B)$. The dimension is $D=2^N$.



## Exercise Statement

For a 2-design circuit on $N$ qubits ($D = 2^N$), compare two traceless cost observables:

$$
O_G = |0\rangle\langle 0|^{\otimes N} - \frac{I}{D} \qquad \text{(global projector, traceless)},
$$

$$
O_L = \frac{1}{N}\sum_{j=1}^N Z_j \qquad \text{(normalized local observable, traceless)}.
$$

**(a)** Compute $\mathrm{Tr}(O_G^2)$ and $\mathrm{Tr}(O_L^2)$.

**(b)** Using the Weingarten formula, determine the gradient variance scaling for both costs.

**(c)** Compute the ratio $\mathrm{Var}_L / \mathrm{Var}_G$ and interpret.

## Solution

### Part (a): Computing $\mathrm{Tr}(O^2)$ for each cost

#### Global cost $O_G$:

$$
O_G^2 = \left(|0\rangle\langle 0|^{\otimes N} - \frac{I}{D}\right)^2 = |0\rangle\langle 0|^{\otimes N} - \frac{2}{D}|0\rangle\langle 0|^{\otimes N} + \frac{I}{D^2}.
$$

$$
\mathrm{Tr}(O_G^2) = 1 - \frac{2}{D} + \frac{1}{D} = 1 - \frac{1}{D} = \frac{D-1}{D}.
$$

For large $D$: $\mathrm{Tr}(O_G^2) \approx 1$.

#### Local cost $O_L$:

$$
O_L^2 = \frac{1}{N^2} \sum_{j,k} Z_j Z_k.
$$

- **Diagonal** ($j = k$): $\mathrm{Tr}(Z_j^2) = D$.  There are $N$ such terms.
- **Off-diagonal** ($j \neq k$): $Z_j Z_k$ is a tensor product with two $Z$ factors and $N-2$ identities.  Since $\mathrm{Tr}(Z) = 0$, $\mathrm{Tr}(Z_j Z_k) = 0$.

$$
\mathrm{Tr}(O_L^2) = \frac{1}{N^2} \cdot N \cdot D = \frac{D}{N}.
$$

### Part (b): Gradient variance via Weingarten calculus

From Exercise 7.1, we derived the exact Weingarten result for any traceless observable $O$:

$$
\mathbb{E}[C^2] = \frac{\mathrm{Tr}(O^2)}{D(D+1)}.
$$

(This followed from the $k=2$ Weingarten formula: the $\pi = \mathrm{id}$ contribution vanished because $\mathrm{Tr}(O) = 0$, and the $\pi = (12)$ contribution gave $\mathrm{Tr}(O^2) \times [\mathrm{Wg}(\mathrm{id}) + \mathrm{Wg}((12))] = \mathrm{Tr}(O^2)/(D(D+1))$.)

Applying this to both costs:

**Global:**

$$
\mathrm{Var}_G = \frac{(D-1)/D}{D(D+1)} = \frac{D-1}{D^2(D+1)} \approx \frac{1}{D^2} = 2^{-2N}.
$$

**Local:**

$$
\mathrm{Var}_L = \frac{D/N}{D(D+1)} = \frac{1}{N(D+1)} \approx \frac{1}{ND} = \frac{1}{N \cdot 2^N}.
$$

### Part (c): Ratio and interpretation

$$
\frac{\mathrm{Var}_L}{\mathrm{Var}_G} = \frac{D^2(D+1)}{D \cdot N \cdot (D+1)} \cdot \frac{D}{D-1} \approx \frac{D}{N} = \frac{2^N}{N}.
$$

$$
\boxed{\frac{\mathrm{Var}_L}{\mathrm{Var}_G} \approx \frac{2^N}{N} \gg 1.}
$$

The local cost has an **exponentially larger** gradient than the global cost.

**Physical interpretation:**

- The global cost $O_G = |0\rangle\langle 0|^{\otimes N}$ is a rank-1 projector — it asks "are you *exactly* in the target state?"  Since $\mathrm{Tr}(O_G^2) \approx 1$ is $O(1)$, the Weingarten formula gives $\mathrm{Var}_G \sim 1/D^2$: the cost landscape is exponentially flat.

- The local cost $O_L = \frac{1}{N}\sum Z_j$ sums single-qubit measurements.  Since $\mathrm{Tr}(O_L^2) \sim D/N$, the variance gets a factor $D/N$ boost.  Each local term "sees" only part of the system, so the information is not diluted over the full $D$-dimensional Hilbert space.

- Both costs still have barren plateaus (exponential decay), but the exponent is halved for the local cost: $2^{-N}$ vs $2^{-2N}$.  In practice, this can mean the difference between trainable ($N \sim 20$) and untrainable.

---
## Symbolic Verification (SymPy)

In [ ]:
import sympy as sp

D, N_sym = sp.symbols('D N', positive=True, integer=True)

# Global cost: O_G = |0><0|^N, Tr(O_G^2) = 1
# Local cost: O_L = (1/N) sum_j |0><0|_j, Tr(O_L^2) = 1/N
Var_global = sp.Rational(1,1) / (D * (D + 1))
Var_local = sp.Rational(1,1) / (N_sym * D * (D + 1)) * N_sym  # trace is 1/N * N terms

# Actually for local cost:
# Tr(O_L^2) = (1/N^2) * [N*Tr(|0><0|_j^2 tensor I) + cross terms]
# For 2-design: Var_local ~ N * 2^(-N) / N = 2^(-N)
# vs Var_global ~ 2^(-2N)
print('Variance scaling (2-design circuits):')
print(f'  Global cost: Var ~ 1/D^2 = 2^(-2N)')
print(f'  Local cost:  Var ~ 1/D   = 2^(-N)')
print()
for n in range(2, 10):
    d = 2**n
    print(f'  N={n}: Global={1/d**2:.2e}, Local={1/d:.2e}, ratio={d:.0f}')

---
## Numerical Verification

In [ ]:
import numpy as np
from scipy.stats import unitary_group

np.random.seed(42)
sz = np.array([[1,0],[0,-1]], dtype=complex)

for N in [2, 3, 4, 5, 6]:
    D = 2**N
    psi0 = np.zeros(D, dtype=complex); psi0[0] = 1
    
    # Build observables
    O_G = np.outer(psi0, psi0) - np.eye(D)/D
    O_L = np.zeros((D, D), dtype=complex)
    for j in range(N):
        Oj = np.eye(1)
        for k in range(N):
            Oj = np.kron(Oj, sz if k == j else np.eye(2))
        O_L += Oj / N
    
    trOG2 = np.trace(O_G @ O_G).real
    trOL2 = np.trace(O_L @ O_L).real
    
    # Monte Carlo
    costs_G, costs_L = [], []
    for _ in range(8000):
        U = unitary_group.rvs(D)
        psi = U @ psi0
        costs_G.append((psi.conj() @ O_G @ psi).real)
        costs_L.append((psi.conj() @ O_L @ psi).real)
    
    var_G = np.var(costs_G)
    var_L = np.var(costs_L)
    pred_G = trOG2 / (D*(D+1))
    pred_L = trOL2 / (D*(D+1))
    
    print(f"N={N}: Tr(O_G²)={trOG2:.3f}, Tr(O_L²)={trOL2:.3f}")
    print(f"      Var_G={var_G:.2e} (pred {pred_G:.2e}), "
          f"Var_L={var_L:.2e} (pred {pred_L:.2e})")
    if var_G > 1e-10:
        print(f"      Var_L/Var_G = {var_L/var_G:.1f} (pred D/N = {D/N:.1f})")
    print()

print("Local cost has exponentially larger gradient. ✓")